In [6]:
#!/usr/bin/env python3

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system"
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

data_path = PROCESSED_DATA_DIR / "customer_churn_features.csv"

model_data = pd.read_csv(data_path)

print("Shape:", model_data.shape)
display(model_data.head())

Shape: (10000, 41)


,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,AgeMissing,MonthlyChargesInvalid,call_avg,...,data_change_3m,sms_change_3m,complaints_change_3m,call_trend_slope,data_trend_slope,complaints_trend_slope,Churn,FirstObservedUsageMonth,SignupDateInvalid,TenureMonths
0,CUST_0,NaN,Male,Urban,Yearly,48.18,2021-03-20,1,0,492.233333,...,2.206667,2.666667,-2,0.252378,0.219266,-9.440559e-02,0,2023-01-31,0,33.377135
1,CUST_1,NaN,Female,Suburban,Prepaid,38.94,2020-01-25,1,0,452.007500,...,0.160000,11.000000,1,-2.704161,-0.147063,1.748252e-02,0,2023-01-31,0,47.174770
2,CUST_10,NaN,Female,Urban,Prepaid,47.81,2022-04-29,1,0,628.351667,...,4.113333,15.333333,-2,1.261119,0.260524,-5.594406e-02,0,2023-01-31,0,20.072273
3,CUST_100,62.0,Female,Urban,Monthly,29.80,2021-10-14,0,0,534.539167,...,3.256667,-1.000000,-2,2.170874,0.341503,-7.342657e-02,0,2023-01-31,0,26.544021
4,CUST_1000,NaN,Female,Suburban,Monthly,61.62,2018-06-10,1,0,493.881667,...,1.176667,10.666667,0,-8.303916,0.146958,-1.531626e-17,0,2023-01-31,0,66.688568


In [7]:
# Identify target and drop columns

print(model_data.columns.tolist())

print("\nTarget distribution:")

print(model_data["Churn"].value_counts())

print("\nTarget percentage:")
print(
    model_data["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

['CustomerID', 'Age', 'Gender', 'Region', 'ContractType', 'MonthlyCharges', 'SignupDate', 'AgeMissing', 'MonthlyChargesInvalid', 'call_avg', 'call_std', 'call_min', 'call_max', 'data_avg', 'data_std', 'data_min', 'data_max', 'sms_avg', 'sms_std', 'complaints_total', 'complaints_avg', 'complaints_max', 'call_avg_last3m', 'data_avg_last3m', 'sms_avg_last3m', 'complaints_total_last3m', 'call_avg_first3m', 'data_avg_first3m', 'sms_avg_first3m', 'complaints_total_first3m', 'call_change_3m', 'data_change_3m', 'sms_change_3m', 'complaints_change_3m', 'call_trend_slope', 'data_trend_slope', 'complaints_trend_slope', 'Churn', 'FirstObservedUsageMonth', 'SignupDateInvalid', 'TenureMonths']

Target distribution:
Churn
0    9214
1     786
Name: count, dtype: int64

Target percentage:
Churn
0    92.14
1     7.86
Name: proportion, dtype: float64


In [8]:
print(model_data[[
    "SignupDate",
    "FirstObservedUsageMonth"
]].dtypes)

SignupDate                 str
FirstObservedUsageMonth    str
dtype: object


In [9]:
# Identify target and drop columns

target = "Churn"

drop_columns = [
    "CustomerID",
    "SignupDate",
    "FirstObservedUsageMonth",
    target
]

X = model_data.drop(columns=drop_columns)

y = model_data[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 37)
y shape: (10000,)


In [10]:
# Identify feature types
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumber of numeric features:")
print(len(numeric_features))

Categorical features:
['Gender', 'Region', 'ContractType']

Number of numeric features:
34


C:\Users\Windows 10\AppData\Local\Temp\ipykernel_15512\1094432081.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [11]:
# Split the data into train, validation, and test sets
from sklearn.model_selection import train_test_split

# First split:
# 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Second split:
# divide temporary 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (7000, 37)
Validation shape: (1500, 37)
Test shape: (1500, 37)


In [12]:
#check class distribution in train, validation, and test sets

def show_class_distribution(name, y_subset):
    print(
        f"{name}:",
        y_subset.value_counts(normalize=True)
        .mul(100)
        .round(2)
        .to_dict()
    )

show_class_distribution("Train", y_train)
show_class_distribution("Validation", y_val)
show_class_distribution("Test", y_test)

Train: {0: 92.14, 1: 7.86}
Validation: {0: 92.13, 1: 7.87}
Test: {0: 92.13, 1: 7.87}


In [19]:
#Import necessary libraries for preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [18]:
# Identify feature types
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical:", categorical_features)
print("Numeric count:", len(numeric_features))

Categorical: ['Gender', 'Region', 'ContractType']
Numeric count: 34


In [25]:
# Define preprocessing pipelines for numeric features 
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Define preprocessing pipelines for categorical features

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore")
        )
    ]
)

In [29]:
# Combine numeric and categorical pipelines into a single preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [30]:
# Transform the datasets using the preprocessor
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (7000, 42)
Processed validation shape: (1500, 42)
Processed test shape: (1500, 42)


In [31]:
# Check for missing values in the processed datasets
print(
    "Any missing values in processed training data:",
    np.isnan(
        X_train_processed.toarray()
        if hasattr(X_train_processed, "toarray")
        else X_train_processed
    ).any()
)

Any missing values in processed training data: False


**Logistic Regression Model**

In [32]:
#Logistic Regression Model Training

from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg.fit(
    X_train_processed,
    y_train
)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to u

In [34]:
# Evaluate the model on the validation set

y_val_proba = log_reg.predict_proba(
    X_val_processed
)[:, 1]

y_val_pred = (
    y_val_proba >= 0.50
).astype(int)

In [35]:
#calculate and print evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", round(accuracy_score(y_val, y_val_pred),4))

print("Precision:", round(precision_score(y_val,y_val_pred),4))

print("Recall:", round(recall_score(y_val, y_val_pred),4))
    

print("F1:",round(f1_score(y_val, y_val_pred),4))

print("ROC-AUC:",round(roc_auc_score(y_val,y_val_proba),4))

print("PR-AUC:",round(average_precision_score( y_val, y_val_proba),4))

Accuracy: 0.5373
Precision: 0.0727
Recall: 0.4153
F1: 0.1237
ROC-AUC: 0.4715
PR-AUC: 0.0775


In [36]:
cm = confusion_matrix(
    y_val,
    y_val_pred
)

print(cm)

[[757 625]
 [ 69  49]]


In [37]:
print(
    classification_report(
        y_val,
        y_val_pred,
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9165    0.5478    0.6857      1382
           1     0.0727    0.4153    0.1237       118

    accuracy                         0.5373      1500
   macro avg     0.4946    0.4815    0.4047      1500
weighted avg     0.8501    0.5373    0.6415      1500



**Random Forest Model**

In [38]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_processed,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [45]:
y_val_proba_rf = rf_model.predict_proba(
    X_val_processed
)[:, 1]

y_val_pred_rf = (
    y_val_proba_rf >= 0.50
).astype(int)

In [46]:
print(
    "Accuracy:",
    round(accuracy_score(y_val, y_val_pred_rf), 4)
)

print(
    "Precision:",
    round(precision_score(y_val, y_val_pred_rf), 4)
)

print(
    "Recall:",
    round(recall_score(y_val, y_val_pred_rf), 4)
)

print(
    "F1:",
    round(f1_score(y_val, y_val_pred_rf), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_val, y_val_proba_rf), 4)
)

print(
    "PR-AUC:",
    round(average_precision_score(y_val, y_val_proba_rf), 4)
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_pred_rf
    )
)

Accuracy: 0.9213
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.4475


c:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


PR-AUC: 0.0688

Confusion Matrix:
[[1382    0]
 [ 118    0]]


In [47]:
print(
    classification_report(
        y_val,
        y_val_pred_rf,
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9213    1.0000    0.9591      1382
           1     0.0000    0.0000    0.0000       118

    accuracy                         0.9213      1500
   macro avg     0.4607    0.5000    0.4795      1500
weighted avg     0.8489    0.9213    0.8836      1500



c:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Windows 10\Desktop\Job based Project\telecom-churn-decision-system\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted

In [48]:
precision_score(
    y_val,
    y_val_pred_rf,
    zero_division=0
)

0.0

**XGBoost Model**

In [ ]:
#XGBoost Classifier Model Training

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


In [55]:
# Calculate scale_pos_weight for XGBoost
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 11.727272727272727


In [ ]:
#XGBoost Classifier Model Training with scale_pos_weight
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [59]:
xgb_model.fit(
    X_train_processed,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [60]:
# Evaluate the XGBoost model on the validation set
y_val_proba_xgb = xgb_model.predict_proba(
    X_val_processed
)[:, 1]

y_val_pred_xgb = (
    y_val_proba_xgb >= 0.50
).astype(int)

In [61]:
#Evaluate the XGBoost model on the validation set
print(
    "Accuracy:",
    round(accuracy_score(y_val, y_val_pred_xgb), 4)
)

print(
    "Precision:",
    round(
        precision_score(
            y_val,
            y_val_pred_xgb,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall:",
    round(recall_score(y_val, y_val_pred_xgb), 4)
)

print(
    "F1:",
    round(f1_score(y_val, y_val_pred_xgb), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_val, y_val_proba_xgb), 4)
)

print(
    "PR-AUC:",
    round(
        average_precision_score(
            y_val,
            y_val_proba_xgb
        ),
        4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_pred_xgb
    )
)

Accuracy: 0.8187
Precision: 0.0523
Recall: 0.0763
F1: 0.0621
ROC-AUC: 0.4496
PR-AUC: 0.0799

Confusion Matrix:
[[1219  163]
 [ 109    9]]


In [62]:
print(
    classification_report(
        y_val,
        y_val_pred_xgb,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0     0.9179    0.8821    0.8996      1382
           1     0.0523    0.0763    0.0621       118

    accuracy                         0.8187      1500
   macro avg     0.4851    0.4792    0.4808      1500
weighted avg     0.8498    0.8187    0.8337      1500



****Logistic Regression Model**

In [63]:
from lightgbm import LGBMClassifier

In [66]:
# LightGBM Classifier Model Training
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [67]:
#fit the LightGBM model on the training data
lgbm_model.fit(
    X_train_processed,
    y_train
)

[LightGBM] [Info] Number of positive: 550, number of negative: 6450
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007177 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5950
[LightGBM] [Info] Number of data points in the train set: 7000, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


,learning_rate,0.05
,n_estimators,300
,class_weight,'balanced'
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,n_jobs,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000


In [69]:
# Evaluate the LightGBM model on the validation set
y_val_proba_lgbm = lgbm_model.predict_proba(
    X_val_processed
)[:, 1]

y_val_pred_lgbm = (
    y_val_proba_lgbm >= 0.50
).astype(int)

In [70]:
#Evaluate the LightGBM model on the validation set
print(
    "Accuracy:",
    round(accuracy_score(y_val, y_val_pred_lgbm), 4)
)

print(
    "Precision:",
    round(
        precision_score(
            y_val,
            y_val_pred_lgbm,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall:",
    round(recall_score(y_val, y_val_pred_lgbm), 4)
)

print(
    "F1:",
    round(f1_score(y_val, y_val_pred_lgbm), 4)
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            y_val,
            y_val_proba_lgbm
        ),
        4
    )
)

print(
    "PR-AUC:",
    round(
        average_precision_score(
            y_val,
            y_val_proba_lgbm
        ),
        4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_pred_lgbm
    )
)

Accuracy: 0.9087
Precision: 0.0476
Recall: 0.0085
F1: 0.0144
ROC-AUC: 0.4549
PR-AUC: 0.0697

Confusion Matrix:
[[1362   20]
 [ 117    1]]


In [73]:
#Classification report for LightGBM model
print(
    classification_report(
        y_val,
        y_val_pred_lgbm,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0     0.9209    0.9855    0.9521      1382
           1     0.0476    0.0085    0.0144       118

    accuracy                         0.9087      1500
   macro avg     0.4843    0.4970    0.4833      1500
weighted avg     0.8522    0.9087    0.8783      1500



**Signal Diagonis**
